In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os

class SignLanguageTrainer:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        # Map the signs to numbers (250 unique signs)
        self.label_map = {sign: i for i, sign in enumerate(self.df['sign'].unique())}
        self.num_classes = len(self.label_map)

    def load_landmarks(self, parquet_path, max_len=100):
        try:
            data = pd.read_parquet(parquet_path)
            # Select x, y, z and fill missing values with 0
            coords = data[['x', 'y', 'z']].fillna(0).values
            
            # --- PADDING LOGIC ---
            # If too long, cut it. If too short, add zeros.
            if len(coords) > max_len:
                coords = coords[:max_len]
            else:
                pad_width = max_len - len(coords)
                coords = np.pad(coords, ((0, pad_width), (0, 0)), mode='constant')
            return coords
        except Exception as e:
            return np.zeros((max_len, 3))

    def data_generator(self, base_path='.', batch_size=32):
        while True:
            # Shuffle the dataframe every epoch
            self.df = self.df.sample(frac=1).reset_index(drop=True)
            for idx in range(0, len(self.df), batch_size):
                X_batch, y_batch = [], []
                batch_df = self.df.iloc[idx : idx + batch_size]
                
                for _, row in batch_df.iterrows():
                    parquet_file = os.path.join(base_path, row['path'])
                    landmarks = self.load_landmarks(parquet_file)
                    X_batch.append(landmarks)
                    y_batch.append(self.label_map[row['sign']])
                
                yield np.array(X_batch), np.array(y_batch)

    def build_model(self, input_shape=(100, 3)):
        model = tf.keras.Sequential([
            # Input is (Time Steps, 3 Coordinates)
            tf.keras.layers.Input(shape=input_shape),
            tf.keras.layers.LSTM(128, return_sequences=False),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(self.num_classes, activation='softmax')
        ])
        model.compile(optimizer='adam', 
                      loss='sparse_categorical_crossentropy', 
                      metrics=['accuracy'])
        return model

# --- RUNNING THE PROJECT ---
# 1. Initialize
trainer = SignLanguageTrainer('train.csv')

# 2. Build the model (using 100 frames as standard length)
model = trainer.build_model(input_shape=(100, 3))

# 3. Create the generator (points to the folder where you unzipped the data)
train_gen = trainer.data_generator(base_path='.', batch_size=32)

# 4. Train
print("Training is starting...")
model.fit(
    train_gen,
    steps_per_epoch=100, 
    epochs=10
)

# 5. Save
model.save('ML_project.h5')
print("Finished! Model saved as ML_project.h5")

Training is starting...
Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 21s 151ms/step - accuracy: 0.0050 - loss: 5.5214
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 14s 144ms/step - accuracy: 0.0028 - loss: 5.5217
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 15s 150ms/step - accuracy: 0.0041 - loss: 5.5214
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 14s 140ms/step - accuracy: 0.0031 - loss: 5.5213
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 15s 153ms/step - accuracy: 0.0041 - loss: 5.5220
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 17s 166ms/step - accuracy: 0.0053 - loss: 5.5210
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 176ms/step - accuracy: 0.0050 - loss: 5.5212
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 22s 217ms/step - accuracy: 0.0044 - loss: 5.5213
Epoch 9/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 182ms/step - accuracy: 0.0031 - loss: 5.5229
Epoch 10/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 180ms/step - accuracy: 0.0028 - loss: 5.5223


Finished! Model saved as ML_project.h5
